# Phase 1 — Part 4: Theoretical Rigor (Methodology Background)

**Purpose:** Satisfy evaluation criteria for *theoretical rigor* by explaining the **mathematics**, **intuition**, and **fit to this project’s data** (CUAD-style legal clauses, Phase 1 ML pipeline: TF-IDF → SVM / Naive Bayes).

**Alignment:** Use this section in a formal report under *Methodology* or *Theoretical Background*, and/or summarize key bullets in your feature-engineering / modeling notebook where models are introduced.



## 1. TF-IDF Theory

**Role in this project:** TF-IDF turns clause text into **numeric feature vectors** so classical ML models (SVM, Naive Bayes) can learn patterns from vocabulary usage.

**One-sentence intuition:** TF-IDF gives **higher scores** to words that show up **often in this clause** but **not in every clause** in the dataset. Shared boilerplate gets **low** scores; distinctive legal wording gets **high** scores.

### Symbol cheat sheet

| Symbol | Meaning |
|--------|--------|
| **word** | One vocabulary item (or n-gram). |
| **this clause** | The clause we are encoding as numbers. |
| **N** | Total number of training clauses. |
| **df(word)** | Number of clauses that contain the word **at least once**. |
| **log** | Logarithm: turns big ratios into smaller, smoother numbers. |



### 1.1 Term Frequency (TF)

**Question TF answers:** *What fraction of this clause’s tokens is this word?*

> **Same idea without fancy symbols:**  
> **TF(word) = (how many times the word appears in this clause) ÷ (total word count in this clause)**

**Math notation:**

$$
\mathrm{TF}(\text{word}, \text{clause}) = \frac{\text{count of word in clause}}{\text{total tokens in clause}}
$$

**Mini example (you can follow with a calculator):**
- Total tokens in the clause = **100**
- Count of “payment” = **3**
- **TF(“payment”) = 3 ÷ 100 = 0.03** → about **3%** of the clause is that word.



### 1.2 Inverse Document Frequency (IDF)

**Question IDF answers:** *Is this word **common in almost every clause** (not helpful) or **rare** (more distinctive)?*

**Intuition in one breath:**
- Word appears in **many** clauses → **small** IDF → we **down-weight** it.
- Word appears in **few** clauses → **large** IDF → we **up-weight** it.

> **Same idea without fancy symbols:**  
> **IDF(word) = log( N ÷ df(word) )**  
> where **df(word)** = how many clauses contain that word at least once.

**Math notation:**

$$
\mathrm{IDF}(\text{word}) = \log\left(\frac{N}{\mathrm{df}(\text{word})}\right)
$$

**Mini example (illustrative numbers):** Let **N = 1000** clauses.
- “Agreement” in **950** clauses → ratio **1000/950 ≈ 1.05** → **small** IDF after log.
- Rare phrase in **5** clauses → ratio **1000/5 = 200** → **much bigger** before log → **higher** IDF.

*Note:* `sklearn` often uses a **smoothed** IDF so division-by-zero never happens; the **idea** is unchanged.



### 1.3 TF-IDF score (putting TF and IDF together)

> **Same idea without fancy symbols:**  
> **TF-IDF(word, clause) = TF(word, clause) × IDF(word)**  
> (multiply the “how loud in this clause” number by the “how special in the dataset” number)

**Math notation:**

$$
\mathrm{TF\text{-}IDF}(\text{word}, \text{clause}) = \mathrm{TF}(\text{word}, \text{clause}) \times \mathrm{IDF}(\text{word})
$$

**N-grams:** If you use bigrams/trigrams, treat each phrase as one “word”; same formulas apply.


## 2. Why TF-IDF for *This* Dataset (EDA → Theory)

**Domain fact:** Legal contracts and CUAD-style clauses contain **repetitive formal vocabulary** (“shall,” “party,” “agreement,” “hereunder,” etc.) that appears in **many** clause types.

**EDA connection:** If exploratory analysis shows **high repetition** of function words and boilerplate across labels, raw word counts would let those frequent-but-uninformative terms dominate. **IDF down-weights** terms that appear in many documents, while **TF** still captures what is locally emphasized in a given clause.

**Why this matches the task:** Clause classification depends on **discriminative legal phrases** (e.g., payment timing, termination triggers, confidentiality scope) more than on generic contract language. TF-IDF is a standard, interpretable way to highlight **label-relevant** vocabulary in **high-dimensional sparse** text features.


## 3. Support Vector Machine (SVM) Theory

**Goal:** Find a **flat boundary** (line in 2D, plane in 3D, **hyperplane** in high dimension) that **separates clause classes** using TF-IDF features.

### 3.1 From “weighted sum” to the hyperplane equation

Each clause becomes a long vector of numbers **x** = (x₁, x₂, …, xₘ) — one entry per vocabulary term (TF-IDF weight).

Training learns a **weight** for each feature **w** = (w₁, w₂, …, wₘ) and a **bias** **b** (a single number that shifts the boundary).

**Score for one clause** = multiply feature by weight for every dimension, then add bias:

> **Same idea without fancy symbols:**  
> **score = (w₁×x₁) + (w₂×x₂) + … + (wₘ×xₘ) + b**

The **decision boundary** is where that score equals **zero**.

**Compact math (what appears in textbooks and rubrics):**

$$
\mathbf{w}^\top \mathbf{x} + b = 0
$$

Here **wᵀx** is just shorthand for the long sum **w₁x₁ + w₂x₂ + … + wₘxₘ**.

**How classification uses this:**
- For a **two-class** story: **positive score** vs **negative score** picks the side of the boundary.
- For **many clause types**, software builds **several** boundaries (e.g. one-vs-rest); the same **linear score** idea repeats.

### 3.2 Margin — why SVM is special

If many boundaries could separate the classes, SVM picks the one with the **widest gap** (margin) between the boundary and the **closest** training points. **Wider gap → usually more stable** predictions when the text is a bit noisy.


## 4. Why SVM for *This* Problem (TF-IDF Features)

**Core link:** TF-IDF (especially with n-grams) creates **very many dimensions** and **mostly zeros** in each vector (**sparse**).

**Why SVM fits:** Linear SVMs are a standard choice for **sparse high-dimensional** text features: training focuses on **hard examples near the boundary** (**support vectors**) instead of treating every training clause equally.

**Phase 1 match:** We stay with **classical** ML—TF-IDF features plus a **linear** kernel SVM learns the **w** and **b** from Section 3 efficiently on sparse matrices.


## 5. Naive Bayes (Baseline)

### 5.1 What we want vs what Bayes gives us

**What we want:** *Given this clause’s features **x**, how probable is each label **y**?*

**Bayes’ rule** rewrites that in terms of things we can estimate from training counts:

> **Same idea without fancy symbols:**  
> **P(label | features) = P(features | label) × P(label) ÷ P(features)**

**Math notation:**

$$
P(y \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y)\, P(y)}{P(\mathbf{x})}
$$

| Piece | Plain meaning |
|--------|----------------|
| **P(y \| x)** | Probability of class **y** **after** seeing **x** — **what we use to classify**. |
| **P(x \| y)** | How likely this **x** is **if** the true class were **y**. |
| **P(y)** | How often class **y** appears in training (class prior). |
| **P(x)** | Same for all classes when comparing; often dropped in “pick the biggest score” steps. |

**Practical decision:** choose the class **y** that makes **P(x|y) × P(y)** largest (or **log P(x|y) + log P(y)** for numerical stability).

### 5.2 The “naive” trick (independence)

Estimating **P(x|y)** for a long vector **x** is hard. Naive Bayes assumes: **given the label, each feature is independent.**

> **Same idea without fancy symbols:**  
> **P(all features | y) ≈ P(feature₁|y) × P(feature₂|y) × … × P(featureₘ|y)**

**Math notation:**

$$
P(\mathbf{x}\mid y) = \prod_{j} P(x_j \mid y)
$$

Words in real clauses are **not** truly independent, but this model is still a **standard fast baseline**.

### 5.3 Why we include it in Agastya Phase 1

Naive Bayes is **simple and quick** to train. Comparing it to SVM shows whether **margin-based** learning adds value on top of your TF-IDF features.


## 6. Evaluation Metrics (Precision, Recall, F1) — and *Why* for Imbalanced Data

### 6.1 Confusion table for **one** class (e.g. Payment)

|  | **Predicted: this class** | **Predicted: not this class** |
|--|---------------------------|-------------------------------|
| **Truth: this class** | **TP** (hit) | **FN** (missed) |
| **Truth: not this class** | **FP** (false alarm) | TN |

### 6.2 Precision, recall, F1

> **Precision (plain):** Of all clauses we **called** this class, what **fraction** were correct?  
> **Precision = TP ÷ (TP + FP)**

$$
\text{Precision} = \frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}}
$$

> **Recall (plain):** Of all clauses that **truly were** this class, what fraction did we **catch**?  
> **Recall = TP ÷ (TP + FN)**

$$
\text{Recall} = \frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}}
$$

> **F1 (plain):** One number that **penalizes** being good at precision **or** recall but terrible at the other (harmonic mean).  
> **F1 = 2 × Precision × Recall ÷ (Precision + Recall)**

$$
F_1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

**Worked micro-example:** TP = 8, FP = 2, FN = 2  
- Precision = 8/(8+2) = **0.8**  
- Recall = 8/(8+2) = **0.8**  
- F1 = **0.8**

### 6.3 Macro vs micro

- **Macro-F1:** compute F1 **per class**, then **average** without favoring big classes — good when labels are **imbalanced**.
- **Micro:** pools all examples — frequent classes can **dominate**.

### 6.4 Why not rely on accuracy alone

If EDA shows **strong class imbalance**, a model can score **high accuracy** by almost always predicting the majority label. **Macro-F1**, plus per-class precision/recall, shows whether **rare clause types** are learned—not just the common ones.


## 7. Loss Function and Optimization (SVM — Bonus Depth)

**Training goal (plain language):**  
Find **w** and **b** so that **most training clauses** are on the **correct side** of the margin, while keeping **w** from growing huge (regularization). Wrong-side or too-close points pay a **penalty**; comfortably correct points pay **zero** penalty in the classic hinge setup.

> **Hinge loss (idea):** If the true label is **+1** or **−1**, the loss is **zero** when **label × score ≥ 1**, otherwise it grows as the margin condition is violated.

**Compact form (optional):** with label $y_i \in \{+1,-1\}$ and score $s_i = \mathbf{w}^\top \mathbf{x}_i + b$:

$$
\text{loss}_i = \max\bigl(0,\; 1 - y_i \, s_i\bigr)
$$

**Read it:** If **yᵢ × sᵢ** is **at least 1**, the **max** picks **0** (no penalty). If not, the loss is **how far below 1** you are.

This ties back to Section 3: minimizing hinge loss **pushes** toward a **large-margin** separator when the data allow it.


## 8. One-Paragraph “Executive Summary” (Paste-Ready)

Each clause is encoded as a **sparse TF-IDF vector**: words frequent **in the clause** but rare **across the corpus** get larger weights, which helps under repetitive legal boilerplate (as EDA suggests). A **linear SVM** learns weights so a **weighted sum of features plus bias** separates classes while favoring a **wide margin**, which suits **high-dimensional sparse** text inputs. **Naive Bayes** supplies a **simple probabilistic baseline** using Bayes’ rule and a **naive independence** shortcut. Because labels can be **imbalanced**, we stress **precision, recall, and macro-F1** instead of **accuracy alone**.


## 9. Checklist vs Rubric-Style Expectations

- [ ] TF-IDF intuition + **TF**, **IDF**, **TF-IDF** formulas  
- [ ] Explicit **legal text / EDA** motivation for TF-IDF  
- [ ] SVM: hyperplane equation + margin intuition + high-dimensional sparse justification  
- [ ] Naive Bayes: Bayes formula + independence assumption + baseline role  
- [ ] Metrics: precision/recall/F1 + **imbalance** justification for macro-F1  
- [ ] Optional+: hinge loss / margin optimization (1–3 lines)


## 10. Implementation Fidelity Note (If Using scikit-learn)

When writing formally, add one sentence stating you used library defaults (or cite them), e.g. TF-IDF smoothing/normalization settings and SVM regularization parameter \(C\), multi-class strategy, and random seed for splits—**theory stays the same**, but reproducibility markers strengthen methodology writing.
